# Silver Layer Orchestrator – dim_products

**Purpose:**  
Enterprise-grade orchestration for Silver layer Product Dimension. Ensures contract-driven execution, quality gates, and operational observability. No business logic, only orchestration.

**Flow:**
1. **Transformation** (`run_transformation_dim_products`) – Cleans, standardizes, and enriches product data. Returns contract JSON.
2. **Validation** (`run_validation_dim_products`) – Applies business-driven data quality rules. Returns contract JSON.
3. **Monitoring** (`run_monitoring_dim_products`) – Interprets validation metrics, evaluates health, and detects trends. Returns contract JSON.

**Tables involved:**
- `workspace.silver.dim_products` (transformation output)
- `workspace.silver.validation_dim_products_results` (record-level validation)
- `workspace.silver.validation_dim_products_metrics` (aggregated validation metrics)
- `workspace.silver.monitoring_dim_products` (monitoring output)
- `workspace.silver.orchestration_log_dim_products` (orchestration log)

**Architecture:** Medallion, Databricks Serverless, Unity Catalog

**Quality Gates:**
- ✅ Transformation must return `status='SUCCESS'` and `records_written > 0`
- ✅ Validation: `valid_percentage >= quality_threshold` (configurable)
- ⚠️ Monitoring: Logs warning if `has_critical_alert=True`

**Exit Codes:**
- `0`: SUCCESS
- `1`: TRANSFORMATION_FAILED
- `2`: VALIDATION_FAILED (quality gate)
- `3`: MONITORING_FAILED

**Design Principles:**
- No business logic in orchestrator
- No .collect(), .count(), cache(), persist()
- No mutation of Silver data outside scripts
- All thresholds hardcoded for auditability (see script docstrings)
- All outputs are Delta tables, partitioned for performance

**References:**
- See transformation_dim_products.py, validation_dim_products.py, monitoring_dim_products.py for business logic and technical details.


In [ ]:
import logging
from datetime import datetime
import uuid
import json
from pyspark.sql import SparkSession, functions as F

# Configure logging
logger = logging.getLogger("silver_dim_products_orchestrator")
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info('=' * 80)
logger.info('SILVER LAYER ORCHESTRATOR - dim_products')
logger.info('=' * 80)

In [ ]:
# Create widgets for configuration (Databricks only)
try:
    dbutils.widgets.text('quality_threshold', '95.0', 'Min Valid %')
    dbutils.widgets.dropdown('fail_on_quality_gate', 'true', ['true', 'false'], 'Fail on Quality Gate')
    QUALITY_THRESHOLD = float(dbutils.widgets.get('quality_threshold'))
    FAIL_ON_QUALITY_GATE = dbutils.widgets.get('fail_on_quality_gate').lower() == 'true'
    logger.info('Configuration loaded from widgets:')
except Exception:
    QUALITY_THRESHOLD = 95.0
    FAIL_ON_QUALITY_GATE = True
    logger.info('Configuration using defaults (no widgets available):')

logger.info(f'  - quality_threshold: {QUALITY_THRESHOLD}%')
logger.info(f'  - fail_on_quality_gate: {FAIL_ON_QUALITY_GATE}')

In [ ]:
# Get or create Spark session
spark = SparkSession.getActiveSession()
if spark is None:
    logger.info('No active SparkSession found, creating new one...')
    spark = SparkSession.builder.appName('Silver_dim_products_orchestrator').getOrCreate()
else:
    logger.info('Using active SparkSession')
logger.info(f'Spark version: {spark.version}')

In [ ]:
# Import Orchestration Functions
import sys
sys.path.append('../dimension_products/')
try:
    from transformation_dim_products import run_transformation_dim_products
    from validation_dim_products import run_validation_dim_products
    from monitoring_dim_products import run_monitoring_dim_products
    logger.info('✅ Successfully imported orchestration functions')
except ImportError as e:
    logger.error(f'❌ Failed to import: {e}')
    raise

In [ ]:
# Initialize execution context
run_id = str(uuid.uuid4())
execution_timestamp = datetime.utcnow()
summary = {
    'run_id': run_id,
    'execution_timestamp': execution_timestamp.isoformat(),
    'transformation': {},
    'validation': {},
    'monitoring': {},
    'status': 'STARTED',
    'error_message': None,
    'exit_code': 0
}

In [ ]:
logger.info('')
logger.info('=' * 80)
logger.info('STEP 1/3: SILVER TRANSFORMATION')
logger.info('=' * 80)
try:
    transformation_result = run_transformation_dim_products(spark=spark)
    # Validate contract
    if not isinstance(transformation_result, dict):
        raise Exception('Transformation contract not a dict')
    required_fields = ['status', 'source_table', 'target_table', 'records_read', 'records_written', 'duration_seconds', 'execution_timestamp']
    for field in required_fields:
        if field not in transformation_result:
            raise Exception(f'Missing field in transformation contract: {field}')
    summary['transformation'] = transformation_result
    logger.info(f"Transformation Status: {transformation_result['status']}")
    logger.info(f"Records Read: {transformation_result.get('records_read', 'N/A')}")
    logger.info(f"Records Written: {transformation_result.get('records_written', 'N/A')}")
    logger.info(f"Duration: {transformation_result.get('duration_seconds', 'N/A')}s")
    if transformation_result['status'] != 'SUCCESS':
        error_val = transformation_result.get('error_message', 'Unknown error')
        error_msg = f'Transformation failed: {error_val}'
        logger.error(f'❌ {error_msg}')
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 1
        raise Exception(error_msg)
    if transformation_result.get('records_written', 0) == 0:
        error_msg = 'Transformation wrote zero records'
        logger.error(f'❌ {error_msg}')
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 1
        raise Exception(error_msg)
    logger.info('✅ Transformation completed successfully')
except Exception as e:
    logger.error(f'❌ Transformation step failed: {str(e)}', exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 1
    raise

In [ ]:
logger.info('')
logger.info('=' * 80)
logger.info('STEP 2/3: SILVER VALIDATION')
logger.info('=' * 80)
try:
    validation_result = run_validation_dim_products(spark=spark)
    if not isinstance(validation_result, dict):
        raise Exception('Validation contract not a dict')
    required_fields = ['status', 'records_validated', 'quality_metrics', 'duration_seconds']
    for field in required_fields:
        if field not in validation_result:
            raise Exception(f'Missing field in validation contract: {field}')
    summary['validation'] = validation_result
    valid_percentage = float(validation_result['quality_metrics'].get('data_quality_score', 0.0))
    logger.info(f"Data Quality Score: {valid_percentage:.2f}%")
    logger.info(f"Critical Violations: {validation_result['quality_metrics'].get('critical_violations', 'N/A')}")
    logger.info(f"Major Violations: {validation_result['quality_metrics'].get('major_violations', 'N/A')}")
    logger.info(f"Minor Violations: {validation_result['quality_metrics'].get('minor_violations', 'N/A')}")
    logger.info(f"Business Key Violations: {validation_result['quality_metrics'].get('business_key_violations', 'N/A')}")
    logger.info(f"Domain Violations: {validation_result['quality_metrics'].get('domain_violations', 'N/A')}")
    logger.info(f"Logic Violations: {validation_result['quality_metrics'].get('logic_violations', 'N/A')}")
    logger.info(f"Temporal Violations: {validation_result['quality_metrics'].get('temporal_violations', 'N/A')}")
    logger.info('🚦 QUALITY GATE CHECK')
    logger.info(f'   Threshold: {QUALITY_THRESHOLD}%')
    logger.info(f'   Actual: {valid_percentage:.2f}%')
    if validation_result['status'] != 'SUCCESS':
        error_val = validation_result.get('error_message', 'Unknown error')
        error_msg = f'Validation failed: {error_val}'
        logger.error(f'❌ {error_msg}')
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 2
        raise Exception(error_msg)
    if valid_percentage < QUALITY_THRESHOLD:
        error_msg = f'Quality gate failed: {valid_percentage:.2f}% < {QUALITY_THRESHOLD}%'
        logger.error(f'❌ {error_msg}')
        if FAIL_ON_QUALITY_GATE:
            summary['status'] = 'FAILED'
            summary['error_message'] = error_msg
            summary['exit_code'] = 2
            raise Exception(error_msg)
        else:
            logger.warning('⚠️ Quality gate failed but continuing (fail_on_quality_gate=false)')
    else:
        logger.info('✅ Quality gate passed')
    logger.info('✅ Validation completed successfully')
except Exception as e:
    logger.error(f'❌ Validation step failed: {str(e)}', exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 2
    raise

In [ ]:
logger.info('')
logger.info('=' * 80)
logger.info('STEP 3/3: SILVER MONITORING')
logger.info('=' * 80)
try:
    monitoring_result = run_monitoring_dim_products(spark=spark)
    if not isinstance(monitoring_result, dict):
        raise Exception('Monitoring contract not a dict')
    required_fields = ['metrics', 'duration_seconds']
    for field in required_fields:
        if field not in monitoring_result:
            raise Exception(f'Missing field in monitoring contract: {field}')
    summary['monitoring'] = monitoring_result
    metrics = monitoring_result['metrics']
    logger.info(f"Health Status: {metrics.get('dataset_health_status', 'N/A')}")
    logger.info(f"Critical Alert: {metrics.get('has_critical_alert', 'N/A')}")
    logger.info(f"Degraded Alert: {metrics.get('has_degraded_alert', 'N/A')}")
    logger.info(f"Trend Warning: {metrics.get('has_trend_warning', 'N/A')}")
    logger.info(f"Valid % MA7: {metrics.get('valid_pct_ma7', 'N/A')}")
    logger.info(f"Critical MA7: {metrics.get('critical_ma7', 'N/A')}")
    if metrics.get('has_critical_alert', False):
        logger.warning('⚠️' * 40)
        logger.warning(f"CRITICAL ALERT DETECTED: {monitoring_result.get('warnings', [])}")
        logger.warning('⚠️' * 40)
    logger.info('✅ Monitoring completed successfully')
except Exception as e:
    logger.error(f'❌ Monitoring step failed: {str(e)}', exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 3
    raise

In [ ]:
# Mark orchestration as successful if we got here
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, BooleanType
)
import json

def _safe_str(value):
    return json.dumps(value, default=str) if value is not None else None

orchestration_log_schema = StructType([

    StructField("run_id", StringType(), False),
    StructField("execution_timestamp", StringType(), False),
    StructField("status", StringType(), False),
    StructField("exit_code", IntegerType(), False),
    StructField("error_message", StringType(), True),

    # Transformation
    StructField("transformation_status", StringType(), True),
    StructField("transformation_source_table", StringType(), True),
    StructField("transformation_target_table", StringType(), True),
    StructField("transformation_records_read", LongType(), True),
    StructField("transformation_records_written", LongType(), True),
    StructField("transformation_duration_seconds", DoubleType(), True),

    # Validation
    StructField("validation_status", StringType(), True),
    StructField("validation_records_validated", LongType(), True),
    StructField("validation_data_quality_score", DoubleType(), True),
    StructField("validation_critical_violations", IntegerType(), True),
    StructField("validation_major_violations", IntegerType(), True),
    StructField("validation_minor_violations", IntegerType(), True),
    StructField("validation_business_key_violations", IntegerType(), True),
    StructField("validation_domain_violations", IntegerType(), True),
    StructField("validation_logic_violations", IntegerType(), True),
    StructField("validation_temporal_violations", IntegerType(), True),
    StructField("validation_duration_seconds", DoubleType(), True),

    # Monitoring
    StructField("monitoring_health_status", StringType(), True),
    StructField("monitoring_has_critical_alert", BooleanType(), True),
    StructField("monitoring_has_degraded_alert", BooleanType(), True),
    StructField("monitoring_has_trend_warning", BooleanType(), True),
    StructField("monitoring_valid_pct_ma7", DoubleType(), True),
    StructField("monitoring_critical_ma7", DoubleType(), True),
    StructField("monitoring_warnings", StringType(), True),
    StructField("monitoring_duration_seconds", DoubleType(), True),

    # Full JSON
    StructField("full_summary_json", StringType(), False),
])


orchestration_log_row = {
    "run_id": str(summary["run_id"]),
    "execution_timestamp": str(summary["execution_timestamp"]),
    "status": str(summary["status"]),
    "exit_code": int(summary["exit_code"]),
    "error_message": summary.get("error_message"),

    # Transformation
    "transformation_status": summary["transformation"].get("status"),
    "transformation_source_table": summary["transformation"].get("source_table"),
    "transformation_target_table": summary["transformation"].get("target_table"),
    "transformation_records_read": summary["transformation"].get("records_read"),
    "transformation_records_written": summary["transformation"].get("records_written"),
    "transformation_duration_seconds": summary["transformation"].get("duration_seconds"),

    # Validation
    "validation_status": summary["validation"].get("status"),
    "validation_records_validated": summary["validation"].get("records_validated"),
    "validation_data_quality_score": summary["validation"].get("quality_metrics", {}).get("data_quality_score"),
    "validation_critical_violations": summary["validation"].get("quality_metrics", {}).get("critical_violations"),
    "validation_major_violations": summary["validation"].get("quality_metrics", {}).get("major_violations"),
    "validation_minor_violations": summary["validation"].get("quality_metrics", {}).get("minor_violations"),
    "validation_business_key_violations": summary["validation"].get("quality_metrics", {}).get("business_key_violations"),
    "validation_domain_violations": summary["validation"].get("quality_metrics", {}).get("domain_violations"),
    "validation_logic_violations": summary["validation"].get("quality_metrics", {}).get("logic_violations"),
    "validation_temporal_violations": summary["validation"].get("quality_metrics", {}).get("temporal_violations"),
    "validation_duration_seconds": summary["validation"].get("duration_seconds"),

    # Monitoring
    "monitoring_health_status": summary["monitoring"].get("metrics", {}).get("dataset_health_status"),
    "monitoring_has_critical_alert": summary["monitoring"].get("metrics", {}).get("has_critical_alert"),
    "monitoring_has_degraded_alert": summary["monitoring"].get("metrics", {}).get("has_degraded_alert"),
    "monitoring_has_trend_warning": summary["monitoring"].get("metrics", {}).get("has_trend_warning"),
    "monitoring_valid_pct_ma7": summary["monitoring"].get("metrics", {}).get("valid_pct_ma7"),
    "monitoring_critical_ma7": summary["monitoring"].get("metrics", {}).get("critical_ma7"),
    "monitoring_warnings": _safe_str(summary["monitoring"].get("warnings")),
    "monitoring_duration_seconds": summary["monitoring"].get("duration_seconds"),

    # Full summary
    "full_summary_json": _safe_str(summary)
}


try:
    orchestration_log_df = spark.createDataFrame(
        [orchestration_log_row],
        schema=orchestration_log_schema
    )

    orchestration_log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.silver.orchestration_log_dim_products")

    logger.info("✅ Orchestration log persisted successfully")

except Exception as e:
    logger.warning(f"⚠️ Failed to persist orchestration log: {e}")


In [ ]:
# For Databricks Workflows: Exit with status
try:
    exit_payload = {
        'status': summary['status'],
        'exit_code': summary['exit_code'],
        'run_id': summary['run_id'],
        'error_message': summary['error_message'],
        'data_quality_score': summary['validation'].get('quality_metrics', {}).get('data_quality_score'),
        'critical_violations': summary['validation'].get('quality_metrics', {}).get('critical_violations'),
        'health_status': summary['monitoring'].get('metrics', {}).get('dataset_health_status'),
        'has_critical_alert': summary['monitoring'].get('metrics', {}).get('has_critical_alert'),
        'monitoring_warnings': summary['monitoring'].get('warnings', [])
    }
    logger.info(f"Exiting with payload: {exit_payload}")
    dbutils.notebook.exit(json.dumps(exit_payload))
except NameError:
    logger.info('Not in Databricks environment, skipping dbutils.notebook.exit()')
    if summary['status'] == 'FAILED':
        logger.error(f"Orchestration failed with exit code {summary['exit_code']}")
        raise Exception(f"Orchestration failed: {summary['error_message']}")
    else:
        logger.info('✅ Orchestration completed successfully in Jupyter environment')